## Script de automação de Envio dos arquivos entre Databricks e BigQuery


### Observações importantes:
- O primeiro passo é fazer **manualmente** o arquivo bruto para a pasta: 
/Volumes/usuario/usuario_ga4/classificacaoheaders_usuario/

- Importante: Na pasta, o único arquivo que deve estar presente é o arquivo a ser enviado, **quaisquer outros arquivos presentes, deverão ser apagados antes da execução do script**.

- Uma vez feito o upload do arquivo desejado, basta executar o script nas células abaixo.

In [ ]:
# importando as bibliotecas necessárias
from pyspark.sql.types import *
from pyspark.sql.functions import trim, col

# Configurações do PATH do arquivo e tabela destino do BQ
VOLUME_PATH = "/Volumes/usuario/usuario_ga4/classificacaoheaders_usuario/"
BQ_TABLE = "projeto-exemplo.db_usuario.base_classificacao_headers"

### Como funciona o script?
Assim que as constantes são definidas:

**Databricks:** VOLUME_PATH = "/Volumes/usuario/usuario_ga4/classificacaoheaders_usuario/"

**BigQuery:** BQ_TABLE = "projeto-exemplo.db_usuario.base_classificacao_headers"

Basta rodar a célula abaixo para que o script possa analisar a pasta Volumes e identificar o arquivo a ser enviado ao BQ.

### Observações importantes:
- O primeiro passo é fazer **manualmente** o arquivo bruto para a pasta: 
/Volumes/usuario/usuario_ga4/classificacaoheaders_usuario/

- Importante: Na pasta, o único arquivo que deve estar presente é o arquivo a ser enviado, **quaisquer outros arquivos presentes, deverão ser apagados antes da execução do script**.



<img src="Prints das pastas\Volumes Databricks.png" alt="Pasta Volumes Databricks" width="1300">






- Uma vez feito o upload do arquivo desejado, basta executar o script nas células abaixo.





In [ ]:
# Buscando e lendo o arquivo dentro da pasta Volumes
files = dbutils.fs.ls(VOLUME_PATH)

csv_files = [f.path for f in files if f.path.endswith(".csv")]

if len(csv_files) == 0:
    raise Exception(
        "Nenhum arquivo CSV foi encontrado na pasta. "
        "Faça o upload manualmente e execute o script novamente."
    )

if len(csv_files) > 1:
    raise Exception(
        "Mais de um arquivo CSV foi encontrado. "
        "A pasta deve conter apenas um arquivo por execução."
    )

file_path = csv_files[0]

print(f"Arquivo encontrado: {file_path}")

# Definindo o schema da tabela no DF
schema = StructType([
    StructField("Headers", StringType(), True),
    StructField("Tipo", StringType(), True),
    StructField("Classificação", StringType(), True)
])

### Lendo o Script
Assim que o arquivo na pasta Volumes for encontrado, o script vai ler o arquivo, identificar as colunas, linhas e dados presentes, e vai validar todo o esquema antes de seguir com a gravação dos dados dentro do BQ.
Uma pequena prévia da formatação dos dados será gerada para o usuário logo abaixo da célula de leitura, afim de confirmar se está ou não correto antes de enfim gravar os dados dentro da tabela do BQ.

In [ ]:
# Lendo o CSV
df = (
    spark.read
    .option("header", True)
    .option("encoding", "UTF-8")
    .schema(schema)
    .csv(file_path)
)

# Limpeza dos dados
df = (
    df
    .withColumnRenamed("Classificação", "Classificacao")
    .withColumn("Headers", trim(col("Headers")))
    .withColumn("Tipo", trim(col("Tipo")))
    .withColumn("Classificacao", trim(col("Classificacao")))
    .dropDuplicates()
)

# Validando o Schema
print("Schema do DataFrame:")
df.printSchema()

print("Prévia dos dados:")
df.show(10, truncate=False)

### Gravando os dados na tabela do BigQuery
Uma vez em que os dados foram validados, e a prévia foi aprovada pelo usuário, basta executar a célula abaixo para enviar os dados para a tabela do BQ:<br>
**"projeto-exemplo.db_usuario.base_classificacao_headers"**

Assim que a gravação dos dados for concluída, uma mensagem de êxito irá aparecer no fim da célula de execução, ou se algo der errado, uma mensagem de erro também irá informar o usuário para que medidas sejam tomadas.

In [ ]:
# Enviando dados ao BigQuery
try:
    (
        df.write
        .format("bigquery").option("temporaryGcsBucket", "southamerica-east1-b2b-sche-73c6b22c-bucket")

        .option("table", BQ_TABLE)
        .mode("append")
        .save()
    )

    print("Dados enviados ao BigQuery com sucesso.")

    # Apagando arquivo da pasta Volumes
    dbutils.fs.rm(file_path, True)

    print("Arquivo removido da pasta Volumes.")

except Exception as e:

    print("Erro ao enviar dados para o BigQuery.")
    print(str(e))

    raise

#### Se tudo der certo, teremos os nossos dados gravados devidamente no BigQuery:

<img src="Prints das pastas\Dados gravados no BigQuery.png" alt="Pasta Volumes Databricks" width="1300">

### Importante:
Assim que os dados são gravados no BQ, o arquivo dentro da pasta Volumes no Databricks é apagado automaticamente.